# Publication Analysis — Ablation, Baseline Comparison, Statistical Significance & More

This notebook runs **all critical experiments** needed for a top-journal submission of the GLAAM-4X project. It is designed to be run on **Google Colab with GPU** after the main training notebook (`modal_train_glaam4x_v4_kaggle`) has completed Sections 1–3 (data download + parsing).

## What This Notebook Covers

| Section | Experiment | Reviewer Question Answered | Priority |
|---------|-----------|---------------------------|----------|
| **3** | **Ablation Study** (8 variants) | "Does each component actually help?" | 🔴 Critical |
| **4** | **Baseline Comparison** (5 baselines) | "Does GLAAM-4X beat standard methods?" | 🔴 Critical |
| **5** | **Statistical Significance** (DeLong + Bootstrap CI) | "Are the differences real or chance?" | 🔴 Critical |
| **6** | **FLOPs / Parameter Count** comparison | "Is the model efficient?" | 🟢 Nice-to-have |
| **7** | **Failure Case Analysis** | "Where does the model fail and why?" | 🟡 Important |
| **8** | **Cross-Dataset Generalization** | "Does it generalize beyond one dataset?" | 🟡 Important |
| **9** | **Summary Tables & Figures** | Publication-ready output | — |

## How These Experiments Are Related

```
Ablation Study ──┐
                  ├──► Statistical Significance ──► Publication Tables
Baseline Comp ────┘            (DeLong + Bootstrap)        + Figures
                       │
FLOPs / Params ────────┤
Failure Analysis ─────┤
Cross-Dataset ─────────┘
```

## Prerequisites

1. **Run Sections 1–3** of `modal_train_glaam4x_v4_kaggle (1).ipynb` first (installs deps, mounts Drive, downloads + parses Kaggle data into `train_v4.csv`, `val_tune_v4.csv`, `test_v4.csv`)
2. Your Drive must have `models/glaam_4x.py` and `utils/losses.py` (same as training notebook)
3. GPU runtime required (Colab T4 is sufficient)
4. Each ablation/baseline trains for **20 epochs** (reduced from 60 for speed — enough to see clear trends)

## Time Estimate

| Section | Time (T4 GPU) |
|---------|---------------|
| Ablation (8 variants × 20 epochs) | ~4–6 hours |
| Baselines (5 models × 20 epochs) | ~2.5–3.5 hours |
| Statistical significance | ~10 min (no training) |
| FLOPs / params | ~5 min (no training) |
| Failure analysis | ~10 min (no training) |
| Cross-dataset | ~1 hour (inference only) |
| **Total** | **~8–11 hours** |

> 💡 You can run ablation and baseline sections in separate Colab sessions — results are saved to Drive after each variant.

# Section 1: Environment Setup

Mounts Drive, verifies data CSVs exist from the training notebook, imports project code, and sets up the device. **Run this first.**

In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ═══════════════════════════════════════════════════════════════
# Mount Drive & set paths
# ═══════════════════════════════════════════════════════════════
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not on Colab — assuming Drive is already mounted or running locally")

DRIVE_BASE = "/content/drive/MyDrive/Backup/dataset/cataract_detection"
if not os.path.exists(DRIVE_BASE):
    raise FileNotFoundError(f"DRIVE_BASE not found: {DRIVE_BASE}. Mount Drive and check path.")

sys.path.insert(0, DRIVE_BASE)

DATA_DIR = Path("/tmp/data")
DISEASE_NAMES = ['Cataract', 'DR', 'Glaucoma', 'Myopia']
IMG_SIZE = 384
BATCH_SIZE = 32
ABLATION_EPOCHS = 20
WARMUP_EPOCHS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ═══════════════════════════════════════════════════════════════
# Verify data exists from training notebook
# ═══════════════════════════════════════════════════════════════
for csv_name in ["train_v4.csv", "val_tune_v4.csv", "test_v4.csv"]:
    csv_path = DATA_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{csv_path} not found!\nRun Sections 1-3 of the training notebook "
            f"(modal_train_glaam4x_v4_kaggle) first to download and parse data."
        )

train_df = pd.read_csv(DATA_DIR / "train_v4.csv")
val_df = pd.read_csv(DATA_DIR / "val_tune_v4.csv")
test_df = pd.read_csv(DATA_DIR / "test_v4.csv")

# ═══════════════════════════════════════════════════════════════
# Output directories on Drive
# ═══════════════════════════════════════════════════════════════
RESULTS_DIR = Path(DRIVE_BASE) / "publication_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_DIR = RESULTS_DIR / "ablation"
BASELINE_DIR = RESULTS_DIR / "baselines"
SIGNIFICANCE_DIR = RESULTS_DIR / "significance"
FIGURES_DIR = RESULTS_DIR / "figures"
for d in [ABLATION_DIR, BASELINE_DIR, SIGNIFICANCE_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Device: {device}")
print(f"✅ Drive base: {DRIVE_BASE}")
print(f"✅ Data: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")
print(f"✅ Results will be saved to: {RESULTS_DIR}")
print(f"✅ Ablation epochs: {ABLATION_EPOCHS} per variant")

# Section 2: Shared Infrastructure

Dataset class, augmentation, metrics, training loop, and evaluation functions reused across all ablation and baseline experiments. Only the model architecture and loss function change per experiment.

In [ ]:
import cv2
from PIL import Image
from torchvision.transforms import v2
import multiprocessing

NUM_WORKERS = min(2, multiprocessing.cpu_count() - 1)

# ── Gaussian Noise transform ─────────────────────────────────────────────────
class GaussianNoise(torch.nn.Module):
    def __init__(self, std_range=(0.04, 0.2), p=0.3):
        super().__init__()
        self.std_range = std_range
        self.p = p
    def forward(self, img):
        if torch.rand(1).item() > self.p:
            return img
        std = torch.empty(1).uniform_(*self.std_range).item()
        return torch.clamp(img + torch.randn_like(img) * std, 0.0, 1.0)

# ── Augmentation pipelines ────────────────────────────────────────────────────
def get_transforms(img_size, is_train, strong_aug=True):
    base_tail = [
        v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    if not is_train:
        return v2.Compose([v2.Resize((img_size, img_size))] + base_tail), None

    train_t = v2.Compose([
        v2.Resize((img_size, img_size)),
        v2.RandomHorizontalFlip(p=0.5), v2.RandomVerticalFlip(p=0.3),
        v2.RandomApply([v2.RandomChoice([v2.RandomRotation((90,90)), v2.RandomRotation((180,180)), v2.RandomRotation((270,270))])], p=0.3),
        v2.RandomAffine(degrees=15, translate=(0.05,0.05), scale=(0.9,1.1)),
        v2.RandomApply([v2.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
        v2.RandomApply([v2.ColorJitter(hue=0.03, saturation=0.2)], p=0.3),
        v2.RandomApply([v2.GaussianBlur(kernel_size=3)], p=0.2),
        v2.RandomApply([v2.ElasticTransform(alpha=50.0, sigma=5.0)], p=0.1),
    ] + base_tail + [GaussianNoise(p=0.3), v2.RandomErasing(p=0.2, scale=(0.01,0.05), ratio=(0.5,2.0))])

    strong_t = None
    if strong_aug:
        strong_t = v2.Compose([
            v2.Resize((img_size, img_size)),
            v2.RandomHorizontalFlip(p=0.5), v2.RandomVerticalFlip(p=0.3),
            v2.RandomApply([v2.RandomChoice([v2.RandomRotation((90,90)), v2.RandomRotation((180,180)), v2.RandomRotation((270,270))])], p=0.3),
            v2.RandomAffine(degrees=30, translate=(0.1,0.1), scale=(0.8,1.2)),
            v2.RandomApply([v2.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
            v2.RandomApply([v2.ColorJitter(hue=0.05, saturation=0.3)], p=0.4),
            v2.RandomApply([v2.GaussianBlur(kernel_size=5)], p=0.3),
            v2.RandomApply([v2.ElasticTransform(alpha=100.0, sigma=5.0)], p=0.2),
        ] + base_tail + [GaussianNoise(std_range=(0.08,0.3), p=0.4), v2.RandomErasing(p=0.3, scale=(0.02,0.08), ratio=(0.5,2.0))])
    return train_t, strong_t

# ── Dataset ────────────────────────────────────────────────────────────────────
class UnifiedDataset(Dataset):
    def __init__(self, df, img_dir, disease_cols, img_size, is_train=False,
                 strong_aug=True, minority_aug_prob=0.7):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.disease_cols = disease_cols
        self.minority_aug_prob = minority_aug_prob if strong_aug else 0.0
        self.transform, self.strong_aug = get_transforms(img_size, is_train, strong_aug)

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        if not os.path.isabs(path):
            for p in [os.path.join(self.img_dir, path), os.path.join(self.img_dir, "raw", path)]:
                if os.path.exists(p):
                    path = p; break
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {path}")
        return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row['image_path'])
        labels = row[self.disease_cols].values.astype(np.float32)
        if self.is_train if hasattr(self, 'is_train') else False:
            pass  # not used in eval mode
        if self.minority_aug_prob > 0 and labels.sum() > 0 and np.random.rand() < self.minority_aug_prob:
            img = self.strong_aug(img)
        else:
            img = self.transform(img)
        return img, torch.tensor(labels)

# ── Metrics ────────────────────────────────────────────────────────────────────
def compute_metrics(logits, labels, thresholds=None):
    probs = 1 / (1 + np.exp(-logits))
    if thresholds is None:
        thresholds = {d: 0.5 for d in DISEASE_NAMES}
    metrics = {}
    for i, disease in enumerate(DISEASE_NAMES):
        y_true, y_prob = labels[:, i], probs[:, i]
        thr = thresholds.get(disease, 0.5)
        y_pred = (y_prob >= thr).astype(int)
        metrics[disease] = {
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5,
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
        }
    metrics['macro_f1'] = np.mean([m['f1'] for m in metrics.values()])
    return metrics, probs

def find_optimal_thresholds(logits, labels):
    probs = 1 / (1 + np.exp(-logits))
    thresholds = {}
    for i, disease in enumerate(DISEASE_NAMES):
        y_true, y_prob = labels[:, i], probs[:, i]
        best_f1, best_thr = 0, 0.5
        for thr in np.arange(0.05, 0.95, 0.01):
            f1 = f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[disease] = float(best_thr)
    return thresholds

# ── Sampler ────────────────────────────────────────────────────────────────────
def get_sample_weights(df, disease_cols):
    pos_counts = df[disease_cols].sum().values
    neg_counts = len(df) - pos_counts
    pos_w = np.sqrt(1.0 / (pos_counts + 1e-6)); pos_w = pos_w / pos_w.sum()
    neg_w = np.sqrt(1.0 / (neg_counts + 1e-6)); neg_w = neg_w / neg_w.sum()
    weights = np.zeros(len(df))
    for i, row in df.iterrows():
        w = sum(pos_w[j] if row[c] == 1 else neg_w[j] for j, c in enumerate(disease_cols))
        weights[i] = w / len(disease_cols)
    return weights

# ── Training & evaluation ─────────────────────────────────────────────────────
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

def train_epoch(model, loader, criterion, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images); loss = criterion(logits, labels)
            if not torch.isfinite(loss): scaler.update(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(images); loss = criterion(logits, labels)
            if not torch.isfinite(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
        total_loss += loss.item()
        all_logits.append(logits.detach().cpu()); all_labels.append(labels.cpu())
    scheduler.step()
    return total_loss / len(loader), torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        logits = model(images.to(device))
        all_logits.append(logits.cpu()); all_labels.append(labels)
    return torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy()

# ── Unified experiment runner ─────────────────────────────────────────────────
def run_experiment(name, model, criterion, save_dir, strong_aug=True, use_warmup=True,
                   epochs=ABLATION_EPOCHS, extra_info=None):
    """Train one variant, evaluate on test, save results to Drive."""
    print(f"\n{'='*60}\n  EXPERIMENT: {name}\n{'='*60}")
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,} (trainable: {n_trainable:,})")

    model = model.to(device)

    # Data loaders
    train_ds = UnifiedDataset(train_df, str(DATA_DIR), DISEASE_NAMES, IMG_SIZE, is_train=True, strong_aug=strong_aug)
    val_ds   = UnifiedDataset(val_df, str(DATA_DIR), DISEASE_NAMES, IMG_SIZE, is_train=False)
    test_ds  = UnifiedDataset(test_df, str(DATA_DIR), DISEASE_NAMES, IMG_SIZE, is_train=False)

    sw = get_sample_weights(train_df, DISEASE_NAMES)
    sampler = WeightedRandomSampler(torch.tensor(sw, dtype=torch.double), len(train_df)*2, replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                              pin_memory=True, prefetch_factor=2, persistent_workers=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # Optimizer & scheduler
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=5e-4)
    warmup = WARMUP_EPOCHS if use_warmup else 0
    def lr_lambda(epoch):
        if epoch < warmup: return float(epoch + 1) / float(max(1, warmup))
        return 0.5 * (1.0 + np.cos(np.pi * (epoch - warmup) / max(1, epochs - warmup)))
    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    # Training loop
    best_val_f1, best_epoch, best_thr = 0.0, 0, {d: 0.5 for d in DISEASE_NAMES}
    history = {'epoch': [], 'train_loss': [], 'val_macro_f1': []}

    for epoch in range(1, epochs + 1):
        t_loss, _, _ = train_epoch(model, train_loader, criterion, optimizer, scheduler)
        v_logits, v_labels = evaluate(model, val_loader)
        v_metrics, _ = compute_metrics(v_logits, v_labels)
        opt_thr = find_optimal_thresholds(v_logits, v_labels)
        v_metrics_opt, _ = compute_metrics(v_logits, v_labels, opt_thr)
        history['epoch'].append(epoch)
        history['train_loss'].append(t_loss)
        history['val_macro_f1'].append(v_metrics_opt['macro_f1'])
        if v_metrics_opt['macro_f1'] > best_val_f1:
            best_val_f1 = v_metrics_opt['macro_f1']; best_epoch = epoch; best_thr = opt_thr
        if epoch % 5 == 0 or epoch == epochs:
            print(f"  Ep {epoch}/{epochs} | Loss {t_loss:.4f} | Val F1 {v_metrics_opt['macro_f1']:.4f} (best {best_val_f1:.4f})")

    # Test evaluation
    t_logits, t_labels = evaluate(model, test_loader)
    test_metrics, test_probs = compute_metrics(t_logits, t_labels, best_thr)

    print(f"\n  ✅ {name} | Best Val F1: {best_val_f1:.4f} (ep {best_epoch}) | Test Macro F1: {test_metrics['macro_f1']:.4f}")
    for d in DISEASE_NAMES:
        m = test_metrics[d]; print(f"    {d:12s} AUC={m['auc']:.4f} F1={m['f1']:.4f}")

    # Save results
    result = {
        'name': name, 'n_params': n_params, 'n_trainable': n_trainable,
        'best_val_f1': best_val_f1, 'best_epoch': best_epoch,
        'test_macro_f1': test_metrics['macro_f1'], 'test_metrics': test_metrics,
        'best_thresholds': best_thr, 'history': history,
        'extra_info': extra_info or {},
    }
    # Save test logits/labels for significance testing
    result['test_logits'] = t_logits.tolist()
    result['test_labels'] = t_labels.tolist()
    result['test_probs'] = test_probs.tolist()

    save_path = save_dir / f"{name.replace(' ', '_')}.json"
    with open(save_path, 'w') as f:
        json.dump(result, f, indent=2, default=str)
    print(f"  Saved: {save_path}")

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

print("✅ Shared infrastructure ready.")

# Section 3: Ablation Study

**Goal:** Systematically remove or replace each GLAAM-4X component to measure its individual contribution.

## Ablation Variants

| # | Name | What Changes | Hypothesis Tested |
|---|------|-------------|-------------------|
| A1 | **Full GLAAM-4X** | Nothing (reference) | Best model — all components |
| A2 | **No MultiScale** | Replace MultiScaleGLAAM (DR) with standard GLAAMBlock | Multi-scale attention helps DR |
| A3 | **No Disease Gating** | Remove gating network, use equal weights | Learned routing helps |
| A4 | **No ASL** | Replace ASL with BCEWithLogitsLoss | ASL > BCE for imbalance |
| A5 | **No Strong Aug** | Disable strong augmentation (70%→0%) | Differential augmentation helps |
| A6 | **No Attention** | All heads = Identity (plain MobileNetV2) | Attention helps at all |
| A7 | **Shared Attention** | One GLAAMBlock for all diseases | Disease-specific > shared |
| A8 | **No Warmup** | Remove warmup, start cosine immediately | Warmup stabilizes training |

Each variant trains for **20 epochs** with the same data, sampler, optimizer, and schedule (except where the ablation modifies the schedule).

In [ ]:
from models.glaam_4x import (
    GLAAM_4X, GLAAMBlock, MultiScaleGLAAM, DiseaseGatingNetwork,
    GlobalAttentionBranch, LocalAttentionBranch, DISEASE_NAMES as GLAAM_DISEASES
)
from utils.losses import AsymmetricLossOptimized

# ═══════════════════════════════════════════════════════════════
# Wrapper: reorders logits [DR, Glaucoma, Cataract, Myopia] → [Cataract, DR, Glaucoma, Myopia]
# ═══════════════════════════════════════════════════════════════
class GLAAM4XWrapper(nn.Module):
    REORDER_IDX = [2, 0, 1, 3]
    def __init__(self, backbone_model):
        super().__init__()
        self.backbone = backbone_model
    def forward(self, x):
        return self.backbone(x)['logits'][:, self.REORDER_IDX]


# ═══════════════════════════════════════════════════════════════
# A1: Full GLAAM-4X (reference — all components)
# ═══════════════════════════════════════════════════════════════
def make_full_glaam4x():
    return GLAAM4XWrapper(GLAAM_4X(pretrained=True, dropout_rate=0.3))

def asl_criterion():
    return AsymmetricLossOptimized(gamma_neg=4.0, gamma_pos=0.0, clip=0.05)

def bce_criterion():
    return nn.BCEWithLogitsLoss()

print("A1 (Full GLAAM-4X) model factory ready.")
print("ASL criterion factory ready.")
print("BCE criterion factory ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# A2: No MultiScale — replace DR's MultiScaleGLAAM with standard GLAAMBlock
# ═══════════════════════════════════════════════════════════════
class GLAAM_4X_NoMultiScale(GLAAM_4X):
    """Same as GLAAM-4X but DR uses standard GLAAMBlock instead of MultiScaleGLAAM."""
    def __init__(self, pretrained=True, dropout_rate=0.3):
        super().__init__(pretrained=pretrained, dropout_rate=dropout_rate)
        # Replace MultiScaleGLAAM with a standard GLAAMBlock (same reduction as Glaucoma)
        self.attention_heads['DR'] = GLAAMBlock(1280, reduction=8)

def make_no_multiscale():
    return GLAAM4XWrapper(GLAAM_4X_NoMultiScale(pretrained=True, dropout_rate=0.3))


# ═══════════════════════════════════════════════════════════════
# A3: No Disease Gating — remove gating network, use equal weights (1/4 each)
# ═══════════════════════════════════════════════════════════════
class GLAAM_4X_NoGating(GLAAM_4X):
    """Same architecture but gating is bypassed (equal weights)."""
    def forward(self, x, return_attention=False):
        features = self.backbone(x)
        specialist_features = {}
        attention_maps = {}
        for disease in GLAAM_DISEASES:
            if disease in self.attention_heads:
                attended = self.attention_heads[disease](features)
            else:
                attended = features
            specialist_features[disease] = F.adaptive_avg_pool2d(attended, 1).flatten(1)
        logits = []
        for disease in GLAAM_DISEASES:
            logits.append(self.classifiers[disease](specialist_features[disease]).squeeze(-1))
        output_logits = torch.stack(logits, dim=1)
        if return_attention:
            return {'logits': output_logits, 'features': specialist_features, 'attention_maps': {}}
        return {'logits': output_logits, 'features': None, 'attention_maps': {}}

def make_no_gating():
    return GLAAM4XWrapper(GLAAM_4X_NoGating(pretrained=True, dropout_rate=0.3))


# ═══════════════════════════════════════════════════════════════
# A6: No Attention — all heads are Identity (plain MobileNetV2 + per-disease classifiers)
# ═══════════════════════════════════════════════════════════════
class GLAAM_4X_NoAttention(GLAAM_4X):
    """No attention heads at all — just backbone + per-disease classifiers."""
    def __init__(self, pretrained=True, dropout_rate=0.3):
        super().__init__(pretrained=pretrained, dropout_rate=dropout_rate)
        self.attention_heads = nn.ModuleDict()  # empty — all Identity

def make_no_attention():
    return GLAAM4XWrapper(GLAAM_4X_NoAttention(pretrained=True, dropout_rate=0.3))


# ═══════════════════════════════════════════════════════════════
# A7: Shared Attention — one GLAAMBlock for all diseases (no specialists)
# ═══════════════════════════════════════════════════════════════
class GLAAM_4X_SharedAttention(GLAAM_4X):
    """All diseases share one GLAAMBlock (reduction=16) instead of specialized heads."""
    def __init__(self, pretrained=True, dropout_rate=0.3):
        super().__init__(pretrained=pretrained, dropout_rate=dropout_rate)
        shared_block = GLAAMBlock(1280, reduction=16)
        self.attention_heads = nn.ModuleDict({
            'DR': shared_block,
            'Glaucoma': shared_block,
            'Cataract': shared_block,
        })

def make_shared_attention():
    return GLAAM4XWrapper(GLAAM_4X_SharedAttention(pretrained=True, dropout_rate=0.3))

print("✅ Ablation model variants defined:")
print("  A2: No MultiScale (DR uses standard GLAAMBlock)")
print("  A3: No Disease Gating (equal weights)")
print("  A6: No Attention (all Identity)")
print("  A7: Shared Attention (one GLAAMBlock for all)")

## Run Ablation Experiments

This cell runs all 8 ablation variants sequentially. Each variant trains for 20 epochs and saves results to Drive. If a Colab session disconnects, already-completed variants are skipped (results loaded from Drive).

> ⏱️ **Time estimate:** ~4–6 hours total on T4 GPU. You can run subsets by commenting out variants.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Define all ablation experiments
# ═══════════════════════════════════════════════════════════════
ABLATION_EXPERIMENTS = [
    # (name, model_factory, criterion_factory, strong_aug, use_warmup, extra_info)
    ("A1_Full_GLAAM4X",      make_full_glaam4x,    asl_criterion, True,  True,  {"removed": "nothing"}),
    ("A2_No_MultiScale",     make_no_multiscale,   asl_criterion, True,  True,  {"removed": "MultiScaleGLAAM for DR"}),
    ("A3_No_Disease_Gating", make_no_gating,       asl_criterion, True,  True,  {"removed": "DiseaseGatingNetwork"}),
    ("A4_No_ASL",            make_full_glaam4x,    bce_criterion, True,  True,  {"removed": "ASL, replaced with BCE"}),
    ("A5_No_Strong_Aug",     make_full_glaam4x,    asl_criterion, False, True,  {"removed": "strong augmentation"}),
    ("A6_No_Attention",      make_no_attention,    asl_criterion, True,  True,  {"removed": "all attention heads"}),
    ("A7_Shared_Attention",  make_shared_attention, asl_criterion, True, True,  {"removed": "disease-specific heads, using shared"}),
    ("A8_No_Warmup",         make_full_glaam4x,    asl_criterion, True,  False, {"removed": "warmup phase"}),
]

# ═══════════════════════════════════════════════════════════════
# Run experiments (skip already-completed ones)
# ═══════════════════════════════════════════════════════════════
ablation_results = {}

for name, model_fn, crit_fn, strong_aug, use_warmup, extra in ABLATION_EXPERIMENTS:
    save_path = ABLATION_DIR / f"{name}.json"
    if save_path.exists():
        print(f"\n✓ {name} already completed — loading from {save_path}")
        with open(save_path) as f:
            ablation_results[name] = json.load(f)
        continue

    print(f"\n{'#'*60}\n# Starting: {name}\n{'#'*60}")
    model = model_fn()
    criterion = crit_fn()
    result = run_experiment(
        name=name, model=model, criterion=criterion,
        save_dir=ABLATION_DIR, strong_aug=strong_aug, use_warmup=use_warmup,
        extra_info=extra)
    ablation_results[name] = result

print(f"\n{'='*60}\n  ABLATION STUDY COMPLETE — {len(ablation_results)} variants\n{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ablation Summary Table + Figure
# ═══════════════════════════════════════════════════════════════
plt.rcParams.update({'font.size': 11, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

# Build summary table
ablation_rows = []
for name, res in ablation_results.items():
    m = res['test_metrics']
    row = {
        'Variant': name,
        'Params (M)': round(res['n_params'] / 1e6, 2),
        'Val F1': round(res['best_val_f1'], 4),
        'Test Macro F1': round(res['test_macro_f1'], 4),
        'Δ F1': round(res['test_macro_f1'] - ablation_results.get('A1_Full_GLAAM4X', {}).get('test_macro_f1', 0), 4),
    }
    for d in DISEASE_NAMES:
        dm = m.get(d, {})
        row[f'{d} AUC'] = round(dm.get('auc', 0), 4)
        row[f'{d} F1'] = round(dm.get('f1', 0), 4)
    ablation_rows.append(row)

ablation_table = pd.DataFrame(ablation_rows)
ablation_table.to_csv(ABLATION_DIR / "ablation_summary.csv", index=False)
print("ABLATION STUDY SUMMARY")
print("=" * 80)
print(ablation_table.to_string(index=False))
print(f"\nSaved: {ABLATION_DIR / 'ablation_summary.csv'}")

# ── Bar chart: Test Macro F1 per variant ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
names = [r['Variant'] for r in ablation_rows]
f1s = [r['Test Macro F1'] for r in ablation_rows]
colors = ['#2196F3'] + ['#FF5722'] * (len(names) - 1)  # highlight full model
bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='none', width=0.6)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=9, rotation=0)
ax.set_ylabel('Test Macro F1', fontsize=12)
ax.set_title('Ablation Study: Test Macro F1 per Variant', fontsize=14, fontweight='bold')
ax.axhline(f1s[0], color='#2196F3', linestyle='--', alpha=0.4, label=f'Full model F1={f1s[0]:.4f}')
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(min(f1s) - 0.03, max(f1s) + 0.03)
fig.savefig(FIGURES_DIR / "ablation_macro_f1.png")
plt.show()

# ── Per-disease AUC heatmap ──────────────────────────────────────────────────
auc_cols = [f'{d} AUC' for d in DISEASE_NAMES]
auc_data = ablation_table[auc_cols].values
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(auc_data, cmap='RdYlGn', aspect='auto', vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(DISEASE_NAMES)))
ax.set_xticklabels(DISEASE_NAMES, fontsize=10)
ax.set_yticks(range(len(names)))
ax.set_yticklabels([n.replace('_', '\n') for n in names], fontsize=8)
for i in range(len(names)):
    for j in range(len(DISEASE_NAMES)):
        ax.text(j, i, f'{auc_data[i,j]:.3f}', ha='center', va='center', fontsize=8,
                color='white' if auc_data[i,j] > 0.8 else 'black')
ax.set_title('Ablation: Per-Disease AUC Heatmap', fontsize=13, fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04, label='AUC')
fig.savefig(FIGURES_DIR / "ablation_auc_heatmap.png")
plt.show()

print(f"\n📊 Figures saved to {FIGURES_DIR}/")

---
# Section 4: Baseline Comparison

**Goal:** Compare GLAAM-4X against standard attention mechanisms and architectures on the **same data, same training pipeline, same evaluation**.

## Baselines

| # | Name | Architecture | Why It's a Standard Baseline |
|---|------|-------------|------------------------------|
| B1 | **Plain MobileNetV2** | MobileNetV2 + linear classifier (no attention) | Backbone-only reference |
| B2 | **SE-Net** | MobileNetV2 + Squeeze-and-Excitation blocks | Most common channel attention |
| B3 | **CBAM** | MobileNetV2 + Convolutional Block Attention Module | Channel + spatial attention (standard) |
| B4 | **ECA-Net** | MobileNetV2 + Efficient Channel Attention | Lightweight attention (ICCV 2020) |
| B5 | **GLAAM-4X (ours)** | MobileNetV2 + disease-specific attention | Our proposed method |

All baselines use the **same backbone** (MobileNetV2), **same loss** (ASL), **same optimizer** (AdamW), **same schedule** (warmup + cosine), **same data** (27,899 images), and **same evaluation** (per-disease thresholds). This ensures a fair comparison — the only variable is the attention mechanism.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Baseline model definitions — all use MobileNetV2 backbone for fair comparison
# ═══════════════════════════════════════════════════════════════

# ── B1: Plain MobileNetV2 (no attention) ─────────────────────────────────────
class PlainMobileNetV2(nn.Module):
    """MobileNetV2 backbone + shared classifier. No attention at all."""
    def __init__(self, n_diseases=4, dropout_rate=0.3):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = mobilenet.features
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate), nn.Linear(256, n_diseases))
    def forward(self, x):
        feat = self.features(x)
        pooled = F.adaptive_avg_pool2d(feat, 1).flatten(1)
        return self.classifier(pooled)

# ── SE (Squeeze-and-Excitation) Block ─────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        w = F.adaptive_avg_pool2d(x, 1).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

# ── B2: MobileNetV2 + SE-Net ─────────────────────────────────────────────────
class MobileNetV2_SE(nn.Module):
    """MobileNetV2 with SE blocks inserted at stages 13 and 17."""
    STAGE_CHANNELS = {13: 96, 17: 320}
    def __init__(self, n_diseases=4, dropout_rate=0.3):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = mobilenet.features
        self.se_blocks = nn.ModuleDict({
            f'se_{s}': SEBlock(c) for s, c in self.STAGE_CHANNELS.items()})
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate), nn.Linear(256, n_diseases))
    def forward(self, x):
        for i, layer in enumerate(self.features):
            x = layer(x)
            key = f'se_{i}'
            if key in self.se_blocks:
                x = self.se_blocks[key](x)
        pooled = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(pooled)

# ── CBAM (Convolutional Block Attention Module) ──────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = self.fc(self.avg_pool(x)); max_ = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg + max_)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, max_], dim=1)))

class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

# ── B3: MobileNetV2 + CBAM ───────────────────────────────────────────────────
class MobileNetV2_CBAM(nn.Module):
    STAGE_CHANNELS = {13: 96, 17: 320}
    def __init__(self, n_diseases=4, dropout_rate=0.3):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = mobilenet.features
        self.cbam_blocks = nn.ModuleDict({
            f'cbam_{s}': CBAMBlock(c) for s, c in self.STAGE_CHANNELS.items()})
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate), nn.Linear(256, n_diseases))
    def forward(self, x):
        for i, layer in enumerate(self.features):
            x = layer(x)
            key = f'cbam_{i}'
            if key in self.cbam_blocks:
                x = self.cbam_blocks[key](x)
        pooled = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(pooled)

# ── ECA (Efficient Channel Attention) ────────────────────────────────────────
class ECABlock(nn.Module):
    """ECA-Net: efficient channel attention via 1D conv (no FC, no reduction)."""
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        w = F.adaptive_avg_pool2d(x, 1).view(b, 1, c)  # (B, 1, C)
        w = self.conv(w).view(b, c, 1, 1)
        return x * self.sigmoid(w)

# ── B4: MobileNetV2 + ECA ────────────────────────────────────────────────────
class MobileNetV2_ECA(nn.Module):
    STAGE_CHANNELS = {13: 96, 17: 320}
    def __init__(self, n_diseases=4, dropout_rate=0.3):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = mobilenet.features
        self.eca_blocks = nn.ModuleDict({
            f'eca_{s}': ECABlock(c) for s, c in self.STAGE_CHANNELS.items()})
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate), nn.Linear(256, n_diseases))
    def forward(self, x):
        for i, layer in enumerate(self.features):
            x = layer(x)
            key = f'eca_{i}'
            if key in self.eca_blocks:
                x = self.eca_blocks[key](x)
        pooled = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(pooled)

# ── B5: GLAAM-4X (ours) — reuse from ablation ────────────────────────────────
# make_full_glaam4x() already defined in Section 3

print("✅ Baseline models defined:")
print("  B1: Plain MobileNetV2 (no attention)")
print("  B2: MobileNetV2 + SE-Net (channel attention)")
print("  B3: MobileNetV2 + CBAM (channel + spatial attention)")
print("  B4: MobileNetV2 + ECA-Net (efficient channel attention)")
print("  B5: GLAAM-4X (ours — disease-specific attention)")

## Run Baseline Experiments

All baselines use the same ASL loss, AdamW optimizer, warmup+cosine schedule, and WeightedRandomSampler — only the attention mechanism differs.

> ⏱️ **Time estimate:** ~2.5–3.5 hours total on T4 GPU.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Define all baseline experiments
# ═══════════════════════════════════════════════════════════════
BASELINE_EXPERIMENTS = [
    # (name, model_factory, criterion_factory, extra_info)
    ("B1_Plain_MobileNetV2", lambda: PlainMobileNetV2(n_diseases=4, dropout_rate=0.3), asl_criterion,
     {"attention": "none", "description": "MobileNetV2 backbone only, no attention"}),
    ("B2_SE_Net", lambda: MobileNetV2_SE(n_diseases=4, dropout_rate=0.3), asl_criterion,
     {"attention": "SE (Squeeze-Excitation)", "description": "Channel attention via GAP + FC"}),
    ("B3_CBAM", lambda: MobileNetV2_CBAM(n_diseases=4, dropout_rate=0.3), asl_criterion,
     {"attention": "CBAM", "description": "Channel + spatial attention"}),
    ("B4_ECA_Net", lambda: MobileNetV2_ECA(n_diseases=4, dropout_rate=0.3), asl_criterion,
     {"attention": "ECA (Efficient Channel Attention)", "description": "1D conv channel attention"}),
    ("B5_GLAAM_4X_Ours", make_full_glaam4x, asl_criterion,
     {"attention": "GLAAM-4X (disease-specific)", "description": "Our proposed method"}),
]

# ═══════════════════════════════════════════════════════════════
# Run baseline experiments (skip completed)
# ═══════════════════════════════════════════════════════════════
baseline_results = {}

for name, model_fn, crit_fn, extra in BASELINE_EXPERIMENTS:
    save_path = BASELINE_DIR / f"{name}.json"
    if save_path.exists():
        print(f"\n✓ {name} already completed — loading from {save_path}")
        with open(save_path) as f:
            baseline_results[name] = json.load(f)
        continue

    print(f"\n{'#'*60}\n# Starting: {name}\n{'#'*60}")
    model = model_fn()
    criterion = crit_fn()
    result = run_experiment(
        name=name, model=model, criterion=criterion,
        save_dir=BASELINE_DIR, strong_aug=True, use_warmup=True,
        extra_info=extra)
    baseline_results[name] = result

print(f"\n{'='*60}\n  BASELINE COMPARISON COMPLETE — {len(baseline_results)} models\n{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Baseline Summary Table + Comparison Figure
# ═══════════════════════════════════════════════════════════════

# Build summary table
baseline_rows = []
for name, res in baseline_results.items():
    m = res['test_metrics']
    row = {
        'Model': name,
        'Attention': res.get('extra_info', {}).get('attention', 'N/A'),
        'Params (M)': round(res['n_params'] / 1e6, 2),
        'Val F1': round(res['best_val_f1'], 4),
        'Test Macro F1': round(res['test_macro_f1'], 4),
    }
    for d in DISEASE_NAMES:
        dm = m.get(d, {})
        row[f'{d} AUC'] = round(dm.get('auc', 0), 4)
        row[f'{d} F1'] = round(dm.get('f1', 0), 4)
    baseline_rows.append(row)

baseline_table = pd.DataFrame(baseline_rows)
baseline_table.to_csv(BASELINE_DIR / "baseline_summary.csv", index=False)
print("BASELINE COMPARISON SUMMARY")
print("=" * 90)
print(baseline_table.to_string(index=False))
print(f"\nSaved: {BASELINE_DIR / 'baseline_summary.csv'}")

# ── Bar chart: Test Macro F1 comparison ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
names = [r['Model'].replace('_', '\n') for r in baseline_rows]
f1s = [r['Test Macro F1'] for r in baseline_rows]
colors = ['#9E9E9E', '#FF9800', '#4CAF50', '#2196F3', '#E91E63']  # last = ours (pink/red)
bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='none', width=0.55)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('Test Macro F1', fontsize=12)
ax.set_title('Baseline Comparison: Test Macro F1', fontsize=14, fontweight='bold')
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003, f'{val:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(min(f1s) - 0.05, max(f1s) + 0.03)
# Highlight our model
bars[-1].set_edgecolor('black'); bars[-1].set_linewidth(2)
fig.savefig(FIGURES_DIR / "baseline_macro_f1.png")
plt.show()

# ── Per-disease AUC grouped bar chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(DISEASE_NAMES))
width = 0.15
for i, (name, res) in enumerate(baseline_results.items()):
    aucs = [res['test_metrics'].get(d, {}).get('auc', 0) for d in DISEASE_NAMES]
    label = name.replace('_', ' ')
    ax.bar(x + i * width, aucs, width, label=label)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(DISEASE_NAMES, fontsize=11)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Baseline Comparison: Per-Disease AUC', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(0.5, 1.01)
fig.savefig(FIGURES_DIR / "baseline_per_disease_auc.png")
plt.show()

print(f"\n📊 Figures saved to {FIGURES_DIR}/")

---
# Section 5: Statistical Significance Testing

**Goal:** Prove that the differences between GLAAM-4X and baselines/ablations are **statistically significant**, not due to chance.

## Tests Performed

| Test | What It Measures | When Used |
|------|-----------------|-----------|
| **DeLong's test** | Statistical significance of AUC difference between two models | Comparing ROC curves |
| **Bootstrap 95% CI** for F1 | Confidence interval for F1 via bootstrap resampling | Reporting F1 with uncertainty |
| **McNemar's test** | Significance of prediction disagreement between two models | Comparing binary predictions |

These tests use the **saved test logits/labels** from Sections 3 and 4 — no additional training needed.

> ⏱️ **Time:** ~10 minutes (pure CPU computation)

In [ ]:
from sklearn.metrics import roc_curve, auc
from scipy import stats

# ═══════════════════════════════════════════════════════════════
# DeLong's test for comparing two AUCs
# Based on: DeLong et al. (1988) "Comparing the areas under two or more
# correlated receiver operating characteristic curves"
# Implementation: compute covariance of AUCs, then z-test
# ═══════════════════════════════════════════════════════════════

def _delong_roc_variance(ground_truth, predictions):
    """Compute variance of AUC via DeLong's method."""
    # Implementation based on the fast DeLong algorithm
    order = np.argsort(-predictions)  # descending
    ground_truth = ground_truth[order]
    predictions = predictions[order]

    pos = ground_truth == 1
    neg = ground_truth == 0
    n_pos = pos.sum()
    n_neg = neg.sum()

    if n_pos == 0 or n_neg == 0:
        return 0.0, 0.0

    # Compute V_10 (positive) and V_01 (negative) components
    tx = np.zeros(n_pos)
    ty = np.zeros(n_neg)
    tz = np.zeros(n_pos + n_neg)

    # Use rank-based approach
    distinct_values = np.unique(predictions)
    # For each positive sample, count negatives ranked below
    v10 = np.zeros(n_pos)
    v01 = np.zeros(n_neg)
    pos_preds = predictions[pos]
    neg_preds = predictions[neg]

    for i, pp in enumerate(pos_preds):
        v10[i] = np.sum(neg_preds < pp) + 0.5 * np.sum(neg_preds == pp)
    for j, np_ in enumerate(neg_preds):
        v01[j] = np.sum(pos_preds > np_) + 0.5 * np.sum(pos_preds == np_)

    v10 /= n_neg
    v01 /= n_pos

    auc_val = v10.mean()

    # Covariance components
    s_10 = np.var(v10, ddof=1) if n_pos > 1 else 0.0
    s_01 = np.var(v01, ddof=1) if n_neg > 1 else 0.0

    var = (s_10 / n_pos) + (s_01 / n_neg)
    return auc_val, var


def delong_test(y_true, y_pred1, y_pred2):
    """
    DeLong's test: compare AUCs of two classifiers on the same data.
    Returns: (auc1, auc2, z_score, p_value)
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred1 = np.asarray(y_pred1, dtype=float)
    y_pred2 = np.asarray(y_pred2, dtype=float)

    auc1, var1 = _delong_roc_variance(y_true, y_pred1)
    auc2, var2 = _delong_roc_variance(y_true, y_pred2)

    # Covariance between the two AUCs (they share the same ground truth)
    # Using the method from DeLong et al.
    n = len(y_true)
    n_pos = y_true.sum()
    n_neg = n - n_pos

    if n_pos == 0 or n_neg == 0:
        return auc1, auc2, 0.0, 1.0

    # Compute covariance
    pos_mask = y_true == 1
    neg_mask = y_true == 0
    pos1 = y_pred1[pos_mask]; neg1 = y_pred1[neg_mask]
    pos2 = y_pred2[pos_mask]; neg2 = y_pred2[neg_mask]

    # V_10 components for covariance
    v10_1 = np.array([np.sum(neg1 < pp) + 0.5 * np.sum(neg1 == pp) for pp in pos1]) / n_neg
    v10_2 = np.array([np.sum(neg2 < pp) + 0.5 * np.sum(neg2 == pp) for pp in pos2]) / n_neg
    v01_1 = np.array([np.sum(pos1 > nn) + 0.5 * np.sum(pos1 == nn) for nn in neg1]) / n_pos
    v01_2 = np.array([np.sum(pos2 > nn) + 0.5 * np.sum(pos2 == nn) for nn in neg2]) / n_pos

    cov_10 = np.cov(v10_1, v10_2, ddof=1)[0, 1] if n_pos > 1 else 0.0
    cov_01 = np.cov(v01_1, v01_2, ddof=1)[0, 1] if n_neg > 1 else 0.0

    cov = (cov_10 / n_pos) + (cov_01 / n_neg)

    # z-test
    diff = auc1 - auc2
    se = np.sqrt(max(var1 + var2 - 2 * cov, 1e-12))
    z = diff / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))  # two-tailed

    return auc1, auc2, z, p_value


# ═══════════════════════════════════════════════════════════════
# Bootstrap 95% CI for F1
# ═══════════════════════════════════════════════════════════════
def bootstrap_f1_ci(y_true, y_pred, n_bootstrap=2000, ci=0.95):
    """Bootstrap confidence interval for F1 score."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    f1s = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        f1s.append(f1_score(y_true[idx], y_pred[idx], zero_division=0))
    f1s = np.array(f1s)
    lower = np.percentile(f1s, (1 - ci) / 2 * 100)
    upper = np.percentile(f1s, (1 + ci) / 2 * 100)
    return f1_score(y_true, y_pred, zero_division=0), lower, upper


# ═══════════════════════════════════════════════════════════════
# McNemar's test for prediction disagreement
# ═══════════════════════════════════════════════════════════════
def mcnemar_test(y_true, pred1, pred2):
    """McNemar's test: are two models' predictions significantly different?"""
    y_true = np.asarray(y_true).astype(int)
    pred1 = np.asarray(pred1).astype(int)
    pred2 = np.asarray(pred2).astype(int)

    correct1 = (pred1 == y_true)
    correct2 = (pred2 == y_true)

    # b: model1 correct, model2 wrong
    # c: model1 wrong, model2 correct
    b = np.sum(correct1 & ~correct2)
    c = np.sum(~correct1 & correct2)

    if b + c == 0:
        return 0.0, 1.0, b, c

    # Use exact binomial test for small samples, chi-square for large
    if b + c < 25:
        # Exact binomial
        p_value = stats.binomtest(min(b, c), b + c, 0.5).pvalue
    else:
        # Chi-square with continuity correction
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1 - stats.chi2.cdf(chi2, 1)

    statistic = (abs(b - c) - 1) ** 2 / (b + c) if (b + c) > 0 else 0.0
    return statistic, p_value, b, c


print("✅ Statistical test functions defined:")
print("  - delong_test(): AUC comparison via DeLong's method")
print("  - bootstrap_f1_ci(): 95% CI for F1 via bootstrap")
print("  - mcnemar_test(): Prediction disagreement via McNemar's test")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run significance tests: GLAAM-4X vs each baseline and ablation
# ═══════════════════════════════════════════════════════════════
np.random.seed(42)

# Load all results (ablation + baseline) that have test_logits/test_labels
all_results = {}
for d in [ABLATION_DIR, BASELINE_DIR]:
    for f in d.glob("*.json"):
        with open(f) as fh:
            r = json.load(fh)
            if 'test_logits' in r and 'test_labels' in r:
                all_results[r['name']] = r

# Reference: our full model
ref_name = "B5_GLAAM_4X_Ours" if "B5_GLAAM_4X_Ours" in all_results else "A1_Full_GLAAM4X"
if ref_name not in all_results:
    raise ValueError(f"Reference model {ref_name} not found in results. Run ablation/baseline first.")

ref = all_results[ref_name]
ref_logits = np.array(ref['test_logits'])
ref_labels = np.array(ref['test_labels'])
ref_probs = 1 / (1 + np.exp(-ref_logits))
ref_thr = ref['best_thresholds']

print(f"Reference model: {ref_name}")
print(f"Test samples: {len(ref_labels)}")
print(f"Reference Test Macro F1: {ref['test_macro_f1']:.4f}\n")

# ── Compare against every other model ─────────────────────────────────────────
significance_rows = []

for name, res in all_results.items():
    if name == ref_name:
        # Bootstrap CI for the reference model itself
        for i, d in enumerate(DISEASE_NAMES):
            y_true = ref_labels[:, i]
            thr = ref_thr.get(d, 0.5)
            y_pred = (ref_probs[:, i] >= thr).astype(int)
            f1, ci_lo, ci_hi = bootstrap_f1_ci(y_true, y_pred)
            significance_rows.append({
                'Comparison': f'{ref_name} vs itself',
                'Disease': d,
                'Ref AUC': roc_auc_score(y_true, ref_probs[:, i]) if len(np.unique(y_true)) > 1 else 0.5,
                'Comp AUC': '—',
                'Δ AUC': 0.0,
                'DeLong z': '—', 'DeLong p': '—',
                'Ref F1': f1, 'F1 95% CI': f'[{ci_lo:.4f}, {ci_hi:.4f}]',
                'McNemar p': '—',
            })
        continue

    comp_logits = np.array(res['test_logits'])
    comp_probs = 1 / (1 + np.exp(-comp_logits))
    comp_thr = res['best_thresholds']

    for i, d in enumerate(DISEASE_NAMES):
        y_true = ref_labels[:, i]
        ref_p = ref_probs[:, i]
        comp_p = comp_probs[:, i]

        # DeLong's test (AUC comparison)
        if len(np.unique(y_true)) > 1:
            auc1, auc2, z, p_delong = delong_test(y_true, ref_p, comp_p)
        else:
            auc1, auc2, z, p_delong = 0.5, 0.5, 0.0, 1.0

        # Bootstrap CI for reference F1
        thr_r = ref_thr.get(d, 0.5)
        thr_c = comp_thr.get(d, 0.5)
        ref_pred = (ref_p >= thr_r).astype(int)
        comp_pred = (comp_p >= thr_c).astype(int)
        f1_ref, ci_lo, ci_hi = bootstrap_f1_ci(y_true, ref_pred)

        # McNemar's test
        mcn_stat, mcn_p, b, c = mcnemar_test(y_true, ref_pred, comp_pred)

        significance_rows.append({
            'Comparison': f'{ref_name} vs {name}',
            'Disease': d,
            'Ref AUC': round(auc1, 4),
            'Comp AUC': round(auc2, 4),
            'Δ AUC': round(auc1 - auc2, 4),
            'DeLong z': round(z, 3),
            'DeLong p': f'{p_delong:.4f}' if p_delong >= 0.0001 else '<0.0001',
            'Ref F1': round(f1_ref, 4),
            'F1 95% CI': f'[{ci_lo:.4f}, {ci_hi:.4f}]',
            'McNemar p': f'{mcn_p:.4f}' if mcn_p >= 0.0001 else '<0.0001',
        })

sig_table = pd.DataFrame(significance_rows)
sig_table.to_csv(SIGNIFICANCE_DIR / "significance_tests.csv", index=False)

print("STATISTICAL SIGNIFICANCE TESTS")
print("=" * 100)
# Print summary: just the key comparisons (vs baselines)
baseline_comparisons = sig_table[sig_table['Comparison'].str.contains('B[1-4]')]
print(baseline_comparisons.to_string(index=False))
print(f"\nFull results saved: {SIGNIFICANCE_DIR / 'significance_tests.csv'}")
print(f"\nTotal comparisons: {len(sig_table)}")
print(f"Significant (p<0.05) DeLong tests: {(sig_table['DeLong p'] != '—').sum() and (sig_table['DeLong p'].apply(lambda x: float(x.replace('<','')) if x != '—' else 1.0) < 0.05).sum()}")
print(f"Significant (p<0.05) McNemar tests: {(sig_table['McNemar p'] != '—').sum() and (sig_table['McNemar p'].apply(lambda x: float(x.replace('<','')) if x != '—' else 1.0) < 0.05).sum()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Significance visualization: p-value heatmap
# ═══════════════════════════════════════════════════════════════

# Extract DeLong p-values for baseline comparisons (matrix: model × disease)
baseline_names = [n for n in all_results.keys() if n.startswith('B') and n != ref_name]
ablation_names = [n for n in all_results.keys() if n.startswith('A') and n != 'A1_Full_GLAAM4X']
comparison_names = baseline_names + ablation_names

if comparison_names:
    # Build p-value matrix
    p_matrix = np.ones((len(comparison_names), len(DISEASE_NAMES)))
    for i, name in enumerate(comparison_names):
        for j, d in enumerate(DISEASE_NAMES):
            row = sig_table[(sig_table['Comparison'] == f'{ref_name} vs {name}') & (sig_table['Disease'] == d)]
            if len(row) > 0:
                p_str = row.iloc[0]['DeLong p']
                if p_str != '—':
                    p_matrix[i, j] = float(p_str.replace('<', ''))

    fig, ax = plt.subplots(figsize=(8, max(4, len(comparison_names) * 0.5)))
    # Use log scale for p-values
    log_p = -np.log10(p_matrix + 1e-10)
    im = ax.imshow(log_p, cmap='YlOrRd', aspect='auto', vmin=0, vmax=5)
    ax.set_xticks(range(len(DISEASE_NAMES)))
    ax.set_xticklabels(DISEASE_NAMES, fontsize=10)
    ax.set_yticks(range(len(comparison_names)))
    ax.set_yticklabels([n.replace('_', '\n') for n in comparison_names], fontsize=8)
    for i in range(len(comparison_names)):
        for j in range(len(DISEASE_NAMES)):
            p = p_matrix[i, j]
            text = f'{p:.3f}' if p >= 0.001 else '<.001'
            color = 'white' if log_p[i, j] > 2.5 else 'black'
            ax.text(j, i, text, ha='center', va='center', fontsize=7, color=color)
    ax.set_title(f"DeLong's Test p-values\n({ref_name} vs each model, -log10 scale)", fontsize=12, fontweight='bold')
    # Add significance threshold line
    ax.axhline(len(baseline_names) - 0.5, color='blue', linewidth=1, linestyle='--', alpha=0.5)
    ax.text(len(DISEASE_NAMES) - 0.5, len(baseline_names) - 0.5, ' ← baselines | ablations →',
            fontsize=7, color='blue', va='center', ha='right')
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04, label='-log10(p)')
    fig.savefig(FIGURES_DIR / "significance_pvalue_heatmap.png")
    plt.show()
    print(f"📊 Saved: {FIGURES_DIR / 'significance_pvalue_heatmap.png'}")

# ── F1 with 95% CI forest plot for GLAAM-4X ───────────────────────────────────
ref_f1_rows = sig_table[sig_table['Comparison'].str.contains('vs itself')]
if len(ref_f1_rows) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    diseases = ref_f1_rows['Disease'].tolist()
    f1s = ref_f1_rows['Ref F1'].tolist()
    cis = ref_f1_rows['F1 95% CI'].tolist()
    ci_los = [float(c.strip('[]').split(',')[0]) for c in cis]
    ci_his = [float(c.strip('[]').split(',')[1]) for c in cis]
    y = range(len(diseases))
    ax.barh(y, f1s, color=['#E74C3C', '#E67E22', '#8E44AD', '#2980B9'], height=0.5, alpha=0.7)
    for i, (f1, lo, hi) in enumerate(zip(f1s, ci_los, ci_his)):
        ax.plot([lo, hi], [i, i], color='black', linewidth=2)
        ax.text(hi + 0.01, i, f'{f1:.3f} [{lo:.3f}, {hi:.3f}]', va='center', fontsize=9)
    ax.set_yticks(list(y))
    ax.set_yticklabels(diseases, fontsize=11)
    ax.set_xlabel('F1 Score with 95% Bootstrap CI', fontsize=11)
    ax.set_title(f'{ref_name}: Per-Disease F1 with 95% CI', fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1.15)
    ax.grid(axis='x', alpha=0.2)
    fig.savefig(FIGURES_DIR / "f1_confidence_intervals.png")
    plt.show()
    print(f"📊 Saved: {FIGURES_DIR / 'f1_confidence_intervals.png'}")

---
# Section 6: FLOPs & Parameter Count Comparison

**Goal:** Show that GLAAM-4X is not only more accurate but also **efficient** — comparable parameter count and FLOPs to standard attention methods.

Measures for each model:
- **Total parameters** (model capacity)
- **Trainable parameters**
- **FLOPs** (computational cost, estimated via `thop`)
- **Inference latency** (single-image forward pass, GPU)

> ⏱️ **Time:** ~5 minutes (no training needed)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FLOPs & Parameter Count Comparison
# ═══════════════════════════════════════════════════════════════
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop"])
from thop import profile, clever_format
import time

# All models to evaluate
efficiency_models = {
    "B1_Plain_MobileNetV2": lambda: PlainMobileNetV2(n_diseases=4, dropout_rate=0.3),
    "B2_SE_Net": lambda: MobileNetV2_SE(n_diseases=4, dropout_rate=0.3),
    "B3_CBAM": lambda: MobileNetV2_CBAM(n_diseases=4, dropout_rate=0.3),
    "B4_ECA_Net": lambda: MobileNetV2_ECA(n_diseases=4, dropout_rate=0.3),
    "B5_GLAAM_4X_Ours": make_full_glaam4x,
}

efficiency_rows = []
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

for name, model_fn in efficiency_models.items():
    model = model_fn().to(device).eval()

    # Parameter count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # FLOPs (via thop)
    try:
        flops, params_thop = profile(model, inputs=(dummy_input,), verbose=False)
        flops_str = clever_format([flops], "%.2f")
    except Exception as e:
        flops = 0; flops_str = f"Error: {e}"

    # Inference latency (single image, GPU)
    timings = []
    with torch.no_grad():
        for _ in range(10):  # warmup
            _ = model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        for _ in range(50):
            t0 = time.time()
            _ = model(dummy_input)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            timings.append((time.time() - t0) * 1000)

    mean_ms = np.mean(timings)
    p95_ms = np.percentile(timings, 95)

    row = {
        'Model': name,
        'Total Params': total_params,
        'Total Params (M)': round(total_params / 1e6, 2),
        'Trainable Params': trainable_params,
        'FLOPs': flops,
        'FLOPs (G)': round(flops / 1e9, 2) if flops > 0 else 0,
        'FLOPs (str)': flops_str,
        'Mean Latency (ms)': round(mean_ms, 2),
        'P95 Latency (ms)': round(p95_ms, 2),
    }
    efficiency_rows.append(row)
    print(f"  {name:25s} | Params: {total_params/1e6:.2f}M | FLOPs: {flops_str} | Latency: {mean_ms:.1f}ms")

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

efficiency_table = pd.DataFrame(efficiency_rows)
efficiency_table.to_csv(RESULTS_DIR / "efficiency_comparison.csv", index=False)
print(f"\n{'='*70}\nEFFICIENCY COMPARISON\n{'='*70}")
print(efficiency_table[['Model', 'Total Params (M)', 'FLOPs (str)', 'Mean Latency (ms)']].to_string(index=False))
print(f"\nSaved: {RESULTS_DIR / 'efficiency_comparison.csv'}")

# ── Efficiency figure: Params vs FLOPs scatter ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for i, row in enumerate(efficiency_rows):
    ax.scatter(row['FLOPs'] / 1e9, row['Total Params (M)'], s=150, zorder=5)
    ax.annotate(row['Model'].replace('_', '\n'), (row['FLOPs'] / 1e9, row['Total Params (M)']),
                fontsize=8, ha='left', va='bottom', xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('FLOPs (G)', fontsize=12)
ax.set_ylabel('Parameters (M)', fontsize=12)
ax.set_title('Efficiency: Parameters vs FLOPs', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
fig.savefig(FIGURES_DIR / "efficiency_scatter.png")
plt.show()
print(f"📊 Saved: {FIGURES_DIR / 'efficiency_scatter.png'}")

---
# Section 7: Failure Case Analysis

**Goal:** Identify and analyze cases where GLAAM-4X fails — false positives and false negatives for each disease. This demonstrates **honest evaluation** and helps reviewers understand the model's limitations.

## Analysis Performed

1. **False Negatives** — disease present but model missed it (high clinical risk)
2. **False Positives** — model flagged disease but it's not present (lower risk, but causes unnecessary referrals)
3. **Borderline cases** — probabilities near threshold (0.4–0.6 range)
4. **Per-source failure rate** — which datasets are hardest?
5. **Confidence calibration of failures** — are failures overconfident?

> ⏱️ **Time:** ~10 minutes (uses saved test logits, no training)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Failure Case Analysis
# ═══════════════════════════════════════════════════════════════

# Load reference model results
ref_data = all_results.get(ref_name)
if ref_data is None:
    raise ValueError(f"Run ablation/baseline experiments first — {ref_name} not found.")

ref_logits = np.array(ref_data['test_logits'])
ref_labels = np.array(ref_data['test_labels'])
ref_probs = 1 / (1 + np.exp(-ref_logits))
ref_thr = ref_data['best_thresholds']

# Get test image paths for failure analysis
test_paths = test_df['image_path'].values
test_sources = test_df.get('source', pd.Series(['unknown'] * len(test_df))).values

failure_rows = []
for i, disease in enumerate(DISEASE_NAMES):
    y_true = ref_labels[:, i].astype(int)
    y_prob = ref_probs[:, i]
    thr = ref_thr.get(disease, 0.5)
    y_pred = (y_prob >= thr).astype(int)

    # Classify each prediction
    tp = (y_pred == 1) & (y_true == 1)  # True positive
    tn = (y_pred == 0) & (y_true == 0)  # True negative
    fp = (y_pred == 1) & (y_true == 0)  # False positive
    fn = (y_pred == 0) & (y_true == 1)  # False negative

    for idx in np.where(fn)[0]:
        failure_rows.append({
            'Disease': disease, 'Type': 'False Negative',
            'Image': test_paths[idx] if idx < len(test_paths) else f'idx_{idx}',
            'Source': test_sources[idx] if idx < len(test_sources) else 'unknown',
            'True Label': 1, 'Predicted': 0,
            'Probability': round(float(y_prob[idx]), 4),
            'Threshold': round(thr, 2),
            'Confidence Gap': round(float(thr - y_prob[idx]), 4),
        })
    for idx in np.where(fp)[0]:
        failure_rows.append({
            'Disease': disease, 'Type': 'False Positive',
            'Image': test_paths[idx] if idx < len(test_paths) else f'idx_{idx}',
            'Source': test_sources[idx] if idx < len(test_sources) else 'unknown',
            'True Label': 0, 'Predicted': 1,
            'Probability': round(float(y_prob[idx]), 4),
            'Threshold': round(thr, 2),
            'Confidence Gap': round(float(y_prob[idx] - thr), 4),
        })

failure_df = pd.DataFrame(failure_rows)
failure_df.to_csv(RESULTS_DIR / "failure_cases.csv", index=False)

# ── Summary statistics ────────────────────────────────────────────────────────
print("FAILURE CASE ANALYSIS")
print("=" * 70)
print(f"\nTotal failures: {len(failure_df)}")
print(f"  False Negatives (missed disease): {(failure_df['Type'] == 'False Negative').sum()}")
print(f"  False Positives (false alarm): {(failure_df['Type'] == 'False Positive').sum()}")

print(f"\nPer-disease failure counts:")
failure_summary = failure_df.groupby(['Disease', 'Type']).size().unstack(fill_value=0)
print(failure_summary)

print(f"\nPer-source failure rates:")
if 'Source' in failure_df.columns:
    src_failures = failure_df.groupby('Source').size().sort_values(ascending=False)
    print(src_failures.head(10))

# ── Borderline cases (prob near threshold) ───────────────────────────────────
print(f"\nBorderline cases (prob within ±0.1 of threshold):")
borderline = failure_df[(failure_df['Confidence Gap'] < 0.1)]
print(f"  {len(borderline)} out of {len(failure_df)} failures are borderline")
if len(borderline) > 0:
    print(borderline[['Disease', 'Type', 'Probability', 'Threshold', 'Confidence Gap']].to_string(index=False))

# ── Visualization: Failure distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: FN vs FP per disease
failure_summary.plot(kind='bar', stacked=True, ax=axes[0],
                     color=['#E74C3C', '#F39C12'], edgecolor='none')
axes[0].set_title('Failure Cases per Disease', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Disease')
axes[0].legend(['False Negative', 'False Positive'])
axes[0].tick_params(axis='x', rotation=0)

# Right: Probability distribution of failures
fn_probs = failure_df[failure_df['Type'] == 'False Negative']['Probability']
fp_probs = failure_df[failure_df['Type'] == 'False Positive']['Probability']
axes[1].hist(fn_probs, bins=20, alpha=0.6, color='#E74C3C', label=f'False Neg (n={len(fn_probs)})')
axes[1].hist(fp_probs, bins=20, alpha=0.6, color='#F39C12', label=f'False Pos (n={len(fp_probs)})')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold ~0.5')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Failure Probability Distribution', fontsize=13, fontweight='bold')
axes[1].legend()

fig.suptitle(f'Failure Case Analysis — {ref_name}', fontsize=14, fontweight='bold', y=1.02)
fig.savefig(FIGURES_DIR / "failure_analysis.png")
plt.show()
print(f"\n📊 Saved: {FIGURES_DIR / 'failure_analysis.png'}")
print(f"📄 Failure cases: {RESULTS_DIR / 'failure_cases.csv'}")

---
# Section 8: Cross-Dataset Generalization

**Goal:** Evaluate GLAAM-4X on each data source **separately** to see if the model generalizes across datasets or overfits to one source.

## Analysis

The test set contains images from multiple sources (ODIR, eye_diseases, DDR, RFMiD, JSIEC, glaucoma_bundle). We split the test predictions by source and compute per-disease AUC/F1 for each.

This answers: **"Does the model work on data it wasn't trained on?"**

> ⏱️ **Time:** ~10 minutes (uses saved test logits, no training)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cross-Dataset Generalization Analysis
# ═══════════════════════════════════════════════════════════════

# Check if test_df has a 'source' column
if 'source' not in test_df.columns:
    # Try to infer source from image path
    def infer_source(path):
        for src in ['odir', 'eye_diseases', 'ddr', 'rfmid', 'jsiec', 'glaucoma_bundle', 'idrid', 'papila', 'refuge2']:
            if src in path.lower():
                return src
        return 'unknown'
    test_sources = test_df['image_path'].apply(infer_source).values
else:
    test_sources = test_df['source'].values

unique_sources = np.unique(test_sources)
print(f"Test set sources: {unique_sources}")
print(f"Source distribution:")
for src in unique_sources:
    count = (test_sources == src).sum()
    print(f"  {src:20s}: {count} images")

# Per-source evaluation
cross_dataset_rows = []
for src in unique_sources:
    mask = test_sources == src
    if mask.sum() < 10:
        print(f"  Skipping {src} (only {mask.sum()} samples)")
        continue

    src_logits = ref_logits[mask]
    src_labels = ref_labels[mask]
    src_metrics, _ = compute_metrics(src_logits, src_labels, ref_thr)

    row = {'Source': src, 'N': int(mask.sum())}
    for d in DISEASE_NAMES:
        m = src_metrics.get(d, {})
        row[f'{d} AUC'] = round(m.get('auc', 0), 4)
        row[f'{d} F1'] = round(m.get('f1', 0), 4)
    row['Macro F1'] = round(src_metrics['macro_f1'], 4)
    cross_dataset_rows.append(row)

cross_dataset_table = pd.DataFrame(cross_dataset_rows)
cross_dataset_table.to_csv(RESULTS_DIR / "cross_dataset_generalization.csv", index=False)

print(f"\n{'='*80}\nCROSS-DATASET GENERALIZATION\n{'='*80}")
print(cross_dataset_table.to_string(index=False))
print(f"\nSaved: {RESULTS_DIR / 'cross_dataset_generalization.csv'}")

# ── Visualization: Per-source Macro F1 ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sources = cross_dataset_table['Source'].tolist()
f1s = cross_dataset_table['Macro F1'].tolist()
colors = plt.cm.Set2(np.linspace(0, 1, len(sources)))
bars = ax.bar(range(len(sources)), f1s, color=colors, edgecolor='none', width=0.6)
ax.set_xticks(range(len(sources)))
ax.set_xticklabels([s.replace('_', '\n') for s in sources], fontsize=9)
ax.set_ylabel('Macro F1', fontsize=12)
ax.set_title('Cross-Dataset Generalization: Test Macro F1 per Source', fontsize=13, fontweight='bold')
for bar, val, n in zip(bars, f1s, cross_dataset_table['N']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}\n(n={n})',
            ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(min(f1s) - 0.1, max(f1s) + 0.1)
fig.savefig(FIGURES_DIR / "cross_dataset_generalization.png")
plt.show()
print(f"📊 Saved: {FIGURES_DIR / 'cross_dataset_generalization.png'}")

---
# Section 9: Publication-Ready Summary Tables & Figures

This section combines all results into the final tables and figures you can directly paste into a journal paper.

## Outputs

1. **Table 1: Baseline Comparison** — GLAAM-4X vs SE-Net vs CBAM vs ECA vs Plain (AUC, F1, params, FLOPs)
2. **Table 2: Ablation Study** — each component's contribution (Δ F1)
3. **Table 3: Statistical Significance** — DeLong p-values + F1 95% CIs
4. **Table 4: Cross-Dataset Generalization** — per-source performance
5. **Table 5: Efficiency** — params, FLOPs, latency
6. **Figure: Combined dashboard** — all key results in one figure

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Publication-Ready Summary
# ═══════════════════════════════════════════════════════════════
plt.rcParams.update({'font.size': 10, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

# ── Table 1: Baseline Comparison (LaTeX-ready) ───────────────────────────────
print("=" * 80)
print("TABLE 1: BASELINE COMPARISON")
print("=" * 80)
if 'baseline_table' in dir():
    table1 = baseline_table[['Model', 'Attention', 'Params (M)', 'Test Macro F1',
                              'Cataract AUC', 'DR AUC', 'Glaucoma AUC', 'Myopia AUC']].copy()
    table1.columns = ['Model', 'Attention', 'Params (M)', 'Macro F1',
                       'Cataract AUC', 'DR AUC', 'Glaucoma AUC', 'Myopia AUC']
    print(table1.to_string(index=False))
    table1.to_csv(RESULTS_DIR / "paper_table1_baselines.csv", index=False)
    # LaTeX
    with open(RESULTS_DIR / "paper_table1_baselines.tex", 'w') as f:
        f.write(table1.to_latex(index=False, float_format="%.4f", caption="Baseline comparison on the held-out test set."))
    print(f"\nSaved: {RESULTS_DIR / 'paper_table1_baselines.csv'} and .tex")

# ── Table 2: Ablation Study (LaTeX-ready) ────────────────────────────────────
print(f"\n{'='*80}\nTABLE 2: ABLATION STUDY\n{'='*80}")
if 'ablation_table' in dir():
    table2 = ablation_table[['Variant', 'Params (M)', 'Test Macro F1', 'Δ F1',
                              'Cataract AUC', 'DR AUC', 'Glaucoma AUC', 'Myopia AUC']].copy()
    print(table2.to_string(index=False))
    table2.to_csv(RESULTS_DIR / "paper_table2_ablation.csv", index=False)
    with open(RESULTS_DIR / "paper_table2_ablation.tex", 'w') as f:
        f.write(table2.to_latex(index=False, float_format="%.4f", caption="Ablation study: contribution of each component."))
    print(f"\nSaved: {RESULTS_DIR / 'paper_table2_ablation.csv'} and .tex")

# ── Table 3: Statistical Significance (key comparisons) ──────────────────────
print(f"\n{'='*80}\nTABLE 3: STATISTICAL SIGNIFICANCE (vs baselines)\n{'='*80}")
if 'sig_table' in dir():
    key_sig = sig_table[sig_table['Comparison'].str.contains('B[1-4]')][
        ['Comparison', 'Disease', 'Ref AUC', 'Comp AUC', 'Δ AUC', 'DeLong p', 'F1 95% CI', 'McNemar p']]
    print(key_sig.to_string(index=False))
    key_sig.to_csv(RESULTS_DIR / "paper_table3_significance.csv", index=False)
    print(f"\nSaved: {RESULTS_DIR / 'paper_table3_significance.csv'}")

# ── Table 4: Cross-Dataset ───────────────────────────────────────────────────
print(f"\n{'='*80}\nTABLE 4: CROSS-DATASET GENERALIZATION\n{'='*80}")
if 'cross_dataset_table' in dir():
    print(cross_dataset_table.to_string(index=False))
    cross_dataset_table.to_csv(RESULTS_DIR / "paper_table4_cross_dataset.csv", index=False)
    print(f"\nSaved: {RESULTS_DIR / 'paper_table4_cross_dataset.csv'}")

# ── Table 5: Efficiency ──────────────────────────────────────────────────────
print(f"\n{'='*80}\nTABLE 5: EFFICIENCY COMPARISON\n{'='*80}")
if 'efficiency_table' in dir():
    table5 = efficiency_table[['Model', 'Total Params (M)', 'FLOPs (str)', 'Mean Latency (ms)', 'P95 Latency (ms)']]
    print(table5.to_string(index=False))
    table5.to_csv(RESULTS_DIR / "paper_table5_efficiency.csv", index=False)
    print(f"\nSaved: {RESULTS_DIR / 'paper_table5_efficiency.csv'}")

# ── Combined Dashboard Figure ────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12), facecolor='white')
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# Panel 1: Baseline Macro F1
ax1 = fig.add_subplot(gs[0, 0])
if 'baseline_table' in dir():
    names = [r['Model'].replace('_', '\n') for r in baseline_rows]
    f1s = [r['Test Macro F1'] for r in baseline_rows]
    colors = ['#9E9E9E', '#FF9800', '#4CAF50', '#2196F3', '#E91E63']
    ax1.bar(range(len(names)), f1s, color=colors, width=0.6)
    ax1.set_xticks(range(len(names))); ax1.set_xticklabels(names, fontsize=7)
    ax1.set_title('Baseline Comparison', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Macro F1'); ax1.grid(axis='y', alpha=0.2)

# Panel 2: Ablation Macro F1
ax2 = fig.add_subplot(gs[0, 1])
if 'ablation_table' in dir():
    names = [r['Variant'].replace('_', '\n') for r in ablation_rows]
    f1s = [r['Test Macro F1'] for r in ablation_rows]
    colors = ['#2196F3'] + ['#FF5722'] * (len(names) - 1)
    ax2.bar(range(len(names)), f1s, color=colors, width=0.6)
    ax2.set_xticks(range(len(names))); ax2.set_xticklabels(names, fontsize=6, rotation=0)
    ax2.set_title('Ablation Study', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Macro F1'); ax2.grid(axis='y', alpha=0.2)

# Panel 3: Efficiency scatter
ax3 = fig.add_subplot(gs[0, 2])
if 'efficiency_table' in dir():
    for row in efficiency_rows:
        ax3.scatter(row['FLOPs'] / 1e9, row['Total Params (M)'], s=100)
        ax3.annotate(row['Model'].split('_')[0], (row['FLOPs'] / 1e9, row['Total Params (M)']),
                     fontsize=7, ha='left', xytext=(3, 3), textcoords='offset points')
    ax3.set_xlabel('FLOPs (G)'); ax3.set_ylabel('Params (M)')
    ax3.set_title('Efficiency', fontsize=11, fontweight='bold')
    ax3.grid(True, alpha=0.3)

# Panel 4: Per-disease AUC (baselines)
ax4 = fig.add_subplot(gs[1, 0])
if 'baseline_table' in dir():
    x = np.arange(len(DISEASE_NAMES)); width = 0.15
    for i, (name, res) in enumerate(baseline_results.items()):
        aucs = [res['test_metrics'].get(d, {}).get('auc', 0) for d in DISEASE_NAMES]
        ax4.bar(x + i * width, aucs, width, label=name.split('_')[0], fontsize=7)
    ax4.set_xticks(x + width * 2); ax4.set_xticklabels(DISEASE_NAMES, fontsize=8)
    ax4.set_title('Per-Disease AUC (Baselines)', fontsize=11, fontweight='bold')
    ax4.set_ylabel('AUC'); ax4.set_ylim(0.5, 1.0); ax4.legend(fontsize=6)
    ax4.grid(axis='y', alpha=0.2)

# Panel 5: Cross-dataset
ax5 = fig.add_subplot(gs[1, 1])
if 'cross_dataset_table' in dir():
    sources = cross_dataset_table['Source'].tolist()
    f1s = cross_dataset_table['Macro F1'].tolist()
    ax5.bar(range(len(sources)), f1s, color=plt.cm.Set2(np.linspace(0, 1, len(sources))), width=0.6)
    ax5.set_xticks(range(len(sources))); ax5.set_xticklabels([s[:8] for s in sources], fontsize=7, rotation=30)
    ax5.set_title('Cross-Dataset Generalization', fontsize=11, fontweight='bold')
    ax5.set_ylabel('Macro F1'); ax5.grid(axis='y', alpha=0.2)

# Panel 6: Failure analysis
ax6 = fig.add_subplot(gs[1, 2])
if 'failure_df' in dir():
    failure_summary = failure_df.groupby(['Disease', 'Type']).size().unstack(fill_value=0)
    if 'False Negative' not in failure_summary.columns:
        failure_summary['False Negative'] = 0
    if 'False Positive' not in failure_summary.columns:
        failure_summary['False Positive'] = 0
    failure_summary = failure_summary[['False Negative', 'False Positive']]
    failure_summary.plot(kind='bar', stacked=True, ax=ax6, color=['#E74C3C', '#F39C12'], edgecolor='none', legend=False)
    ax6.set_title('Failure Cases', fontsize=11, fontweight='bold')
    ax6.set_ylabel('Count'); ax6.tick_params(axis='x', rotation=0)

fig.suptitle('GLAAM-4X Publication Analysis Dashboard', fontsize=16, fontweight='bold', y=0.98)
fig.savefig(FIGURES_DIR / "publication_dashboard.png")
plt.show()
print(f"\n📊 Dashboard saved: {FIGURES_DIR / 'publication_dashboard.png'}")

# ── Final summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  PUBLICATION ANALYSIS COMPLETE")
print(f"{'='*80}")
print(f"\nAll results saved to: {RESULTS_DIR}/")
print(f"\nFiles generated:")
for f in sorted(RESULTS_DIR.rglob("*")):
    if f.is_file():
        rel = f.relative_to(RESULTS_DIR)
        print(f"  {rel}")
print(f"\n{'='*80}")
print(f"  Ready for paper submission!")
print(f"{'='*80}")